<hr>

# ℹ️ DATA EXPLORATION


<style>
h1 {
    text-align: left;
    color: blue;
    font-weight: bold;
}

</style>
<hr>

```text
DATA COLLECTION Steps:

1) Data Exploration
Understand data and take notes:
    
    Data
    - DVF (2020-S2 to 2025-S1) 6 large files in the format of .txt
    - arrondissements.csv
    - communes.csv
    - Amentities Tool https://overpass-turbo.eu/
    - Rent data in/outside Paris APIs https://www.observatoire-des-loyers.fr/

    Data Transformation & Saving
    - DVF 6 TXT files merged  into 1 CSV with selected columns:
    - arrondissements.csv file with seelected columns:
    - communes.csv file with selected columns:
    
    Notes for Data cleaning, Analysis, ML

```

DATA TRANSFORMATION & SAVING
- DVF 6 TXT files merged  into 1 CSV with selected columns:
- arrondissements.csv file with seelected columns:
- communes.csv file with selected columns:

---
<style>
h3 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

### 📂 IMPORTs

In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200) # to display all columns in the dataframe

---
## **💰 VALEURS FONCIERES 2020-S2 _ 2025-S1**

DVF dataset is used for:

- 📈 Price prediction
- 📊 Market trends
- 🤖 ML models

🎯 TARGET VARIABLE = Valeur fonciere (price)

to clearly see:
- 📈 Market growth over 5–6 years
- 🏠 Difference between houses vs apartments
- 📉 Market dips (e.g., COVID effects)

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name             | Python data type       | Definition                                  |
|------------------------|------------------------|-----------------------------------------------------|
| **Date mutation**        | `date`          | transaction date               | 
| **Nature mutation**        | `str`          | transaction nature               | 
| **valeur_fonciere**            | `float`                | transaction value          | 
| **Commune**           | `str`                 | Commune official Name       |
| **Type local**        | `str`          | Property variations: Appartement; Dépendance; Maison; Local industriel, commercial ou assimilé.               |
| **Identifiant local**        | `str`          |                |
| **Surface reelle bati**        | `str`          |                |
| **Nombre pieces principales**        | `int`          | Nombre pieces principales            |
| **Surface terrain**        | `float`          | property surface m2               |             
| **Nombre de lots**        | `int`          |                |
| **Voie**           | `str`                 |           |  
| **No voie**           | `str`                 |        |  
| **B/T/Q**           | `str`                 |          |  
| **code_postale**           | `int`                 |  FR Postal code        |  
| **departement**             | `int`                 | first two digits in the Postal code. Extracted from `code_postale`; a unique identifier of the department region.                      |



<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### ➕ COMBINE - 6 TXT FILES INTO 1 CSV
Drop columns as well

In [2]:
import pandas as pd

# List of DVF files
dvf_files = [
    "../data/raw/ValeursFoncieres-2020-S2.txt",
    "../data/raw/ValeursFoncieres-2021.txt",
    "../data/raw/ValeursFoncieres-2022.txt",
    "../data/raw/ValeursFoncieres-2023.txt",
    "../data/raw/ValeursFoncieres-2024.txt",
    "../data/raw/ValeursFoncieres-2025-S1.txt"
]

# Columns to keep
usecols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Commune",
    "Type local",
    "Identifiant local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Nombre de lots",
    "Surface terrain",
    "No voie",  # street number
    "B/T/Q", # Bâtiment, Type, Quartier
    "Voie", # street name
    "Code postal"


]

# Output CSV
output_file = "../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv"

# Specify dtype for columns that may have mixed types
dtype_dict = {
    "Voie": str,
    "No voie": str,
    "Code voie": str,
    "B/T/Q": str,
    "Type local": str
}

with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    header_written = False

    for file in dvf_files:
        print(f"Processing: {file}")

        for chunk in pd.read_csv(
            file,
            sep="|",
            usecols=usecols,
            chunksize=50_000,
            encoding="latin-1",
            decimal=",",
            low_memory=False,  # safer inference
            dtype=dtype_dict
        ):
            # Fix Type local encoding
            chunk["Type local"] = chunk["Type local"].astype(str).str.encode("latin-1", errors="ignore").str.decode("utf-8", errors="ignore")

            # Convert numeric columns
            chunk["Valeur fonciere"] = pd.to_numeric(chunk["Valeur fonciere"], errors="coerce")
            chunk["Surface reelle bati"] = pd.to_numeric(chunk["Surface reelle bati"], errors="coerce")

            # Drop rows with missing price
            chunk = chunk.dropna(subset=["Valeur fonciere"])

            # Calculate price per m² where surface is available
            chunk["Prix_m2"] = chunk["Valeur fonciere"] / chunk["Surface reelle bati"]
            chunk.loc[chunk["Surface reelle bati"].isna(), "Prix_m2"] = None  # handle missing surface

            # Extract department from postal code
            # chunk["Departement"] = chunk["Code postal"].astype(str).str[:2]

            # Write to CSV
            chunk.to_csv(
                f_out,
                index=False,
                header=not header_written
            )

            header_written = True
            del chunk  # free memory

print(f"✅ File saved at {output_file}")

Processing: ../data/raw/ValeursFoncieres-2020-S2.txt
Processing: ../data/raw/ValeursFoncieres-2021.txt
Processing: ../data/raw/ValeursFoncieres-2022.txt
Processing: ../data/raw/ValeursFoncieres-2023.txt
Processing: ../data/raw/ValeursFoncieres-2024.txt
Processing: ../data/raw/ValeursFoncieres-2025-S1.txt
✅ File saved at ../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DISPLAY - HEAD

In [3]:
import pandas as pd
df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")

# Display df DataFrame
print("DATA EXPLORATION:")
display(df.head(10))
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")

DATA EXPLORATION:


,Date mutation,Nature mutation,Valeur fonciere,No voie,B/T/Q,Voie,Code postal,Commune,Nombre de lots,Type local,Identifiant local,Surface reelle bati,Nombre pieces principales,Surface terrain,Prix_m2
0,01/07/2020,Vente,31234.16,NaN,NaN,SAINT JULIEN,1560.0,SAINT-JULIEN-SUR-REYSSOUZE,0,NaN,NaN,NaN,NaN,1192.0,NaN
1,01/07/2020,Vente,278000.00,NaN,NaN,A LA PEROUSE,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,NaN,10092.0,NaN
2,01/07/2020,Vente,278000.00,NaN,NaN,A LA PEROUSE,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,NaN,4570.0,NaN
3,01/07/2020,Vente,278000.00,NaN,NaN,AUX COMMUNS,1250.0,CORVEISSIAT,0,NaN,NaN,NaN,NaN,5750.0,NaN
4,01/07/2020,Vente,278000.00,NaN,NaN,EN COMBARNAUD,1250.0,SIMANDRE-SUR-SURAN,0,NaN,NaN,NaN,NaN,648170.0,NaN
5,01/07/2020,Vente,278000.00,NaN,NaN,EN COMBARNAUD,1250.0,SIMANDRE-SUR-SURAN,0,NaN,NaN,NaN,NaN,150000.0,NaN
6,01/07/2020,Vente,278000.00,NaN,NaN,A LANSIAT,1250.0,SIMANDRE-SUR-SURAN,0,NaN,NaN,NaN,NaN,1763.0,NaN
7,01/07/2020,Vente,278000.00,NaN,NaN,MOULIN DE CORRERIE,1250.0,SIMANDRE-SUR-SURAN,0,NaN,NaN,NaN,NaN,108.0,NaN
8,01/07/2020,Vente,278000.00,NaN,NaN,COMBE BIZET,1250.0,SIMANDRE-SUR-SURAN,0,NaN,NaN,NaN,NaN,124420.0,NaN
9,01/07/2020,Vente,278000.00,NaN,NaN,AU COMBE BISET,1250.0,SIMANDRE-SUR-SURAN,0,NaN,NaN,NaN,NaN,104426.0,NaN


External Dataset Shape: 19908349 rows and 15 columns



<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DISPLAY - UNIQUE VALUES

In [4]:
# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())

# Loop through each column and print unique values
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")

print(f"\nData types check:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")

Data Types & Missing Values of Each Column:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19908349 entries, 0 to 19908348
Data columns (total 15 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   Date mutation              object 
 1   Nature mutation            object 
 2   Valeur fonciere            float64
 3   No voie                    float64
 4   B/T/Q                      object 
 5   Voie                       object 
 6   Code postal                float64
 7   Commune                    object 
 8   Nombre de lots             int64  
 9   Type local                 object 
 10  Identifiant local          float64
 11  Surface reelle bati        float64
 12  Nombre pieces principales  float64
 13  Surface terrain            float64
 14  Prix_m2                    float64
dtypes: float64(8), int64(1), object(6)
memory usage: 2.2+ GB


None

Column: Date mutation
Unique values (1804): ['01/07/2020' '04/07/2020' '02/07/2020' ... '01/05/2025' '18/05/2025'
 '05/01/2025']

Column: Nature mutation
Unique values (6): ['Vente' "Vente en l'Ã©tat futur d'achÃ¨vement"
 'Vente terrain Ã\xa0 bÃ¢tir' 'Echange' 'Adjudication' 'Expropriation']

Column: Valeur fonciere
Unique values (466667): [3.123416e+04 2.780000e+05 3.985000e+03 ... 7.088090e+05 5.503861e+05
 2.441758e+07]

Column: No voie
Unique values (9121): [  nan  347. 1084. ... 8562. 8301. 8423.]

Column: B/T/Q
Unique values (41): [nan 'B' 'C' 'A' 'F' 'L' 'T' 'X' 'O' 'Z' 'D' 'G' 'Q' 'Y' 'I' 'E' 'N' 'H'
 '1' 'W' 'V' 'U' 'S' 'R' 'P' 'M' 'K' 'J' '0' '5' '9' '8' 'b' '7' '2' '4'
 '3' '6' '-' '.' '*']

Column: Voie
Unique values (1125220): ['SAINT JULIEN' 'A LA PEROUSE' 'AUX COMMUNS' ... 'SAINT PIERRE AMELOT'
 'SAINT HONORE D EYLAU' 'DES FOSSES SAINT MARCEL']

Column: Code postal
Unique values (5872): [ 1560.  1250.  1310. ... 97380. 97314. 97330.]

Column: Commune
Unique values (31569

In [8]:
print("Unique values in 'Type local':")
df["Type local"].unique().tolist()

Unique values in 'Type local':


[nan,
 'Maison',
 'Dépendance',
 'Appartement',
 'Local industriel. commercial ou assimilé']

---
## **🚇 LIGNES DE TRANSPORT EN COMMUN**

TRANSPORT dataset is used for:
- a
- b
- c


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name             | Python Data Type       | Concise Definition                                  |
|------------------------|------------------------|-----------------------------------------------------|

In [ ]:
# LIGNES DE TRANSPORT EN COMMUN
df_lignes_transport = pd.read_csv("../data/raw/traces-des-lignes-de-transport-en-commun-idfm.csv", sep=";")
print("Lignes de transport en commun:")
display(df_lignes_transport.head())

Lignes de transport en commun:


,ID,Short Name,Long Name,Route Type,Color,Route URL,Shape,id_ilico,OperatorName,NetworkName,ID_Bus_Contrat,url,Type,long_name_first,geo_point_2d
0,IDFM:C00029,502,502,Bus,FBE324,NaN,"{""coordinates"": [[[2.605063, 48.801975], [2.60...",C00029,Keolis Grand Paris Vallée de la Marne,Marne et Brie,9.0,https://ilico.iledefrance-mobilites.fr/uploads...,HORAIRE|PLAN,5,"48.78188816972777, 2.6480891283837553"
1,IDFM:C01094,57,57,Bus,6E6E00,NaN,"{""coordinates"": [[[2.409755, 48.863434], [2.41...",C01094,RATP,NaN,NaN,https://www.ratp.fr/sites/default/files/lines-...,HORAIRE|PLAN,5,"48.83515885931267, 2.3687425840691674"
2,IDFM:C02220,Soir,Soir Mennecy,Bus,640082,NaN,NaN,C02220,Keolis Val d'Essonne Deux Vallées,Essonne Sud Est,31.0,https://ilico.iledefrance-mobilites.fr/uploads...,HORAIRE|PLAN,S,NaN
3,IDFM:C00527,6183,6183,Bus,640082,NaN,"{""coordinates"": [[[2.037603, 48.707024], [2.03...",C00527,Keolis Vélizy Vallée de la Bièvre,Vélizy Vallées,27.0,https://ilico.iledefrance-mobilites.fr/uploads...,HORAIRE|PLAN,6,"48.72894729625489, 2.0909603606050835"
4,IDFM:C02301,GPSO Bus,GPSO Bus,Bus,FF9900,NaN,"{""coordinates"": [[[2.182615, 48.825516], [2.18...",C02301,Origami / Mobicité,Grand Paris Seine Ouest,NaN,https://ilico.iledefrance-mobilites.fr/uploads...,HORAIRE|PLAN,G,"48.81986189913045, 2.1925887330951395"


---
## **🚇 APIs**

TRANSPORT dataset is used for:
- a
- b
- c


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🔲 DATASET

| Field Name             | Python Data Type       | Concise Definition                                  |
|------------------------|------------------------|-----------------------------------------------------|

In [ ]:
# api for ameneties
